# 📥 TelegramDL — Setup & Install

Install dependencies and clone the repository.

**Run this cell first!**

In [ ]:
#@title 🔧 Install & Clone { display-mode: "form" }

#@markdown ### Install Dependencies
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

packages = ["kurigram", "tgcrypto", "motor", "Flask", "gunicorn", "nest_asyncio"]
for pkg in packages:
    install(pkg)

print("✅ Dependencies installed!")

#@markdown ### Clone / Update Repository
import os

repo_url = "https://github.com/Shineii86/TelegramDL.git" #@param {type:"string"}
repo_name = "TelegramDL"

if os.path.exists(repo_name):
    print("📥 Updating repository...")
    !cd $repo_name && git pull
else:
    print("📥 Cloning repository...")
    !git clone $repo_url

print("\n✅ Setup complete! Run the next cell.")

# ⚙️ TelegramDL — Configuration

Fill in your credentials and settings below.

**Required:** API_ID, API_HASH, BOT_TOKEN

**Optional:** STRING_SESSION (for restricted content)

In [ ]:
#@title 🔑 Telegram Credentials { display-mode: "form" }
import os

#@markdown ### Required Credentials
API_ID = "" #@param {type:"string"}
API_HASH = "" #@param {type:"string"}
BOT_TOKEN = "" #@param {type:"string"}

#@markdown ### User Session (for restricted content)
STRING_SESSION = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Bot Settings
LOGIN_SYSTEM = False #@param {type:"boolean"}
WAITING_TIME = 10 #@param {type:"integer"}
MAX_FILE_SIZE_MB = 2048 #@param {type:"integer"}
TYPE_FILTER = "all" #@param ["all", "photo", "video", "audio"] {type:"string"}

#@markdown ---
#@markdown ### Advanced Settings
CAPTION_ENABLED = True #@param {type:"boolean"}
KEEP_ORIGINAL_CAPTION = True #@param {type:"boolean"}
FORWARD_MODE = True #@param {type:"boolean"}
USE_CHECKPOINT = True #@param {type:"boolean"}
KEEP_ALIVE = True #@param {type:"boolean"}
KEEP_ALIVE_INTERVAL = 30 #@param {type:"integer"}
SESSION_LIMIT_HOURS = 12 #@param {type:"integer"}

#@markdown ---
#@markdown ### Database (only if LOGIN_SYSTEM=True)
DB_URI = "" #@param {type:"string"}
DB_NAME = "telegramdl" #@param {type:"string"}

#@markdown ---
#@markdown ### Admin Settings
ADMINS = "" #@param {type:"string"}
CHANNEL_ID = "" #@param {type:"string"}

# Set all environment variables
os.environ['API_ID'] = API_ID
os.environ['API_HASH'] = API_HASH
os.environ['BOT_TOKEN'] = BOT_TOKEN
os.environ['STRING_SESSION'] = STRING_SESSION
os.environ['LOGIN_SYSTEM'] = str(LOGIN_SYSTEM).lower()
os.environ['WAITING_TIME'] = str(WAITING_TIME)
os.environ['MAX_FILE_SIZE_MB'] = str(MAX_FILE_SIZE_MB)
os.environ['TYPE_FILTER'] = TYPE_FILTER
os.environ['CAPTION_ENABLED'] = str(CAPTION_ENABLED).lower()
os.environ['KEEP_ORIGINAL_CAPTION'] = str(KEEP_ORIGINAL_CAPTION).lower()
os.environ['FORWARD_MODE'] = str(FORWARD_MODE).lower()
os.environ['USE_CHECKPOINT'] = str(USE_CHECKPOINT).lower()
os.environ['KEEP_ALIVE'] = str(KEEP_ALIVE).lower()
os.environ['KEEP_ALIVE_INTERVAL'] = str(KEEP_ALIVE_INTERVAL)
os.environ['SESSION_LIMIT_HOURS'] = str(SESSION_LIMIT_HOURS)
os.environ['DB_URI'] = DB_URI
os.environ['DB_NAME'] = DB_NAME
os.environ['ADMINS'] = ADMINS
os.environ['CHANNEL_ID'] = CHANNEL_ID
os.environ['OUTPUT_DIR'] = "./downloads"
os.environ['ERROR_MESSAGE'] = "true"

# Validate
errors = []
if not API_ID: errors.append("API_ID")
if not API_HASH: errors.append("API_HASH")
if not BOT_TOKEN: errors.append("BOT_TOKEN")

if errors:
    print(f"❌ Missing required fields: {', '.join(errors)}")
else:
    print("✅ Configuration saved!")
    print(f"\n📋 Settings:")
    print(f"   API_ID: {API_ID[:4]}...{API_ID[-4:]}" if len(API_ID) > 8 else f"   API_ID: {API_ID}")
    print(f"   BOT_TOKEN: {BOT_TOKEN[:10]}...")
    print(f"   STRING_SESSION: {'Set' if STRING_SESSION else 'Not set'}")
    print(f"   LOGIN_SYSTEM: {LOGIN_SYSTEM}")
    print(f"   WAITING_TIME: {WAITING_TIME}s")
    print(f"   MAX_FILE_SIZE_MB: {MAX_FILE_SIZE_MB}MB")
    print(f"   TYPE_FILTER: {TYPE_FILTER}")
    print(f"   FORWARD_MODE: {FORWARD_MODE}")
    print(f"   KEEP_ALIVE: {KEEP_ALIVE}")
    print(f"   CHECKPOINT: {USE_CHECKPOINT}")

# 🚀 TelegramDL — Run Bot

Start the bot. It will keep running until you stop this cell.

**Make sure you've run the Setup and Configuration cells first!**

In [ ]:
#@title 🤖 Start Bot { display-mode: "form" }
import nest_asyncio
nest_asyncio.apply()

import os
import threading
os.chdir('TelegramDL')

# Start Flask keep-alive server
if os.environ.get('KEEP_ALIVE', 'true').lower() == 'true':
    from app import app
    def run_flask():
        app.run(host='0.0.0.0', port=8080, debug=False)
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    print('✅ Flask keep-alive started on port 8080')

# Start keep-alive pinger
if os.environ.get('KEEP_ALIVE', 'true').lower() == 'true':
    from utils.keepalive import KeepAlive
    interval = int(os.environ.get('KEEP_ALIVE_INTERVAL', '30'))
    keepalive = KeepAlive(interval_minutes=interval)
    keepalive.start()

print("🚀 Starting TelegramDL Bot...")
print("   Press Stop to end the bot.")
print("\n" + "="*50)

from bot import main
main()

# 🔐 Generate Session String

Run this cell to generate a session string for restricted content access.

**You need this if:**
- Downloading from private/restricted channels
- Downloading stories
- Downloading from private groups

In [ ]:
#@title 🔑 Session Generator { display-mode: "form" }
import os
import sys

try:
    from pyrogram import Client
except ImportError:
    os.system("pip install kurigram tgcrypto -q")
    from pyrogram import Client

def generate_session():
    print("=" * 50)
    print("  TelegramDL Session String Generator")
    print("=" * 50)
    print()

    api_id = int(input("Enter your API ID: "))
    api_hash = input("Enter your API Hash: ")
    phone = input("Enter your phone number (with country code, e.g. +1234567890): ")

    client = Client(":memory:", api_id=api_id, api_hash=api_hash)
    client.start()

    code = client.send_code(phone)
    code_str = input(f"\nEnter the code sent to {phone} (format: 1 2 3 4 5): ").replace(" ", "")

    try:
        client.sign_in(phone, code.phone_code_hash, code_str)
    except Exception:
        password = input("\nTwo-step verification enabled. Enter your password: ")
        client.check_password(password)

    session_string = client.export_session_string()

    print("\n" + "=" * 50)
    print("  Your Session String:")
    print("=" * 50)
    print(session_string)
    print("=" * 50)
    print("\n📋 Copy this and paste in Configuration cell as STRING_SESSION.")

    client.stop()

generate_session()

# 📊 Quick Status

Run this cell to check bot status and downloaded files.

In [ ]:
#@title 📈 Check Status { display-mode: "form" }
import os

print("=" * 50)
print("  TelegramDL Status")
print("=" * 50)

# Check config
api_id = os.environ.get('API_ID', '')
bot_token = os.environ.get('BOT_TOKEN', '')
session = os.environ.get('STRING_SESSION', '')

print(f"\n📋 Configuration:")
print(f"   API_ID: {'Set' if api_id else 'Not set'}")
print(f"   BOT_TOKEN: {'Set' if bot_token else 'Not set'}")
print(f"   STRING_SESSION: {'Set' if session else 'Not set'}")

# Check downloads
download_dir = "./downloads"
if os.path.exists(download_dir):
    files = []
    for root, dirs, filenames in os.walk(download_dir):
        for f in filenames:
            files.append(os.path.join(root, f))
    total_size = sum(os.path.getsize(f) for f in files)
    print(f"\n📥 Downloads:")
    print(f"   Files: {len(files)}")
    print(f"   Size: {total_size / 1024 / 1024:.1f} MB")
else:
    print(f"\n📥 Downloads: No files yet")

# Check checkpoint
if os.path.exists('checkpoint.json'):
    import json
    with open('checkpoint.json', 'r') as f:
        data = json.load(f)
    downloaded = len(data.get('downloaded', []))
    print(f"\n💾 Checkpoint:")
    print(f"   Downloaded IDs: {downloaded}")
else:
    print(f"\n💾 Checkpoint: No checkpoint found")

print("\n" + "=" * 50)